# Logistic Regression Baseline — EDA

A small exploratory analysis of the multinomial Logistic Regression baseline (`src/models/log_reg.py`):
data exploration, a season-based train/test run, and a few figures (including a scatterplot that shows the
logistic relationship between team-strength gap and predicted home-win probability).

> Scratch / exploration only — not part of the pipeline. Uses a season split (train on earlier seasons,
> test on the latest) so the figures are reasonably honest, but the rigorous metric still comes from the
> walk-forward backtest story.

### Setup — load the feature matrix

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the project root importable (this notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features
from src.models.log_reg import train_log_reg, predict_proba, LABEL_ORDER

X, y, meta = build_features(save_schema=False)
y = y.astype(object)  # numpy-backed labels so boolean masks index numpy arrays cleanly
meta = meta.copy()
meta["date"] = pd.to_datetime(meta["date"])
print(f"X: {X.shape}   y: {len(y)}   seasons: {sorted(meta['season'].unique())}")
X.head()

## Part 1 — Data exploration

### Outcome class balance

In [ ]:
counts = y.value_counts().reindex(LABEL_ORDER)
props = (counts / counts.sum()).round(3)
print(props)

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(['Home win', 'Draw', 'Away win'], counts.values, color=['#2a9d8f', '#e9c46a', '#e76f51'])
ax.set_ylabel('matches')
ax.set_title('Outcome distribution (the bar to beat: always-home = %.1f%%)' % (props['HW'] * 100))
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout(); plt.show()

### Elo gap vs outcome
Does the strength signal separate outcomes? `elo_diff` (home minus away) should skew positive for home wins.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
colors = {'HW': '#2a9d8f', 'D': '#e9c46a', 'AW': '#e76f51'}
for label in LABEL_ORDER:
    ax.hist(X['elo_diff'][y.values == label], bins=30, alpha=0.6, label=label, color=colors[label])
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('elo_diff (home - away)'); ax.set_ylabel('matches')
ax.set_title('Elo gap by actual outcome'); ax.legend()
plt.tight_layout(); plt.show()

### Feature correlations
A heatmap over a readable subset of features — highlights redundancy (many rolling variants move together).

In [ ]:
subset = [c for c in X.columns if any(k in c for k in ['elo', 'position', 'rolling_win_rate', 'points_per_game'])][:12]
corr = X[subset].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(subset))); ax.set_xticklabels(subset, rotation=90, fontsize=7)
ax.set_yticks(range(len(subset))); ax.set_yticklabels(subset, fontsize=7)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title('Feature correlation (subset)')
plt.tight_layout(); plt.show()

## Part 2 — Train & run the model (season split)
Train on the earlier seasons, test on the latest — a simple leakage-safe time split for honest figures.

In [ ]:
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix

test_season = sorted(meta['season'].unique())[-1]
train_mask = meta['season'] < test_season
test_mask = meta['season'] == test_season
X_tr, y_tr = X[train_mask], y[train_mask]
X_te, y_te = X[test_mask], y[test_mask]
print(f"Train: {len(X_tr)} (< {test_season})   Test: {len(X_te)} ({test_season})")

model = train_log_reg(X_tr, y_tr)
proba = predict_proba(model, X_te)            # columns: [p_home, p_draw, p_away]
pred = model.predict(X_te)

ll = log_loss(y_te, model.predict_proba(X_te), labels=model.classes_)
acc = accuracy_score(y_te, pred)
print(f"Test log loss: {ll:.4f}   accuracy: {acc:.3f}   (Elo baseline log loss ~1.00)")

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_te, pred, labels=LABEL_ORDER)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(LABEL_ORDER)
ax.set_yticks(range(3)); ax.set_yticklabels(LABEL_ORDER)
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title('Confusion matrix (test season)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout(); plt.show()
# Note: draws are rarely the top pick for any model -- expected, not a bug.

## Part 3 — Figures: testing the regression

### Scatterplot: strength gap vs predicted home-win probability
The core 'does the regression behave?' plot. Each point is a test match: x = Elo gap, y = the model's
predicted `p_home`. A clean upward logistic S-curve means the model learned that a bigger home Elo edge
→ higher home-win probability. Points are coloured by what actually happened.

In [ ]:
p_home = proba[:, 0]
elo_gap = X_te['elo_diff'].to_numpy()

fig, ax = plt.subplots(figsize=(7, 4.5))
for label in LABEL_ORDER:
    m = y_te.values == label
    ax.scatter(elo_gap[m], p_home[m], s=18, alpha=0.6, label=label, color=colors[label])

# Overlay the fitted trend (sorted by elo_gap) to show the logistic shape.
order = np.argsort(elo_gap)
ax.plot(elo_gap[order], p_home[order], color='k', lw=1.0, alpha=0.5)
ax.axhline(0.5, color='grey', lw=0.7, ls=':')
ax.set_xlabel('elo_diff (home - away)'); ax.set_ylabel('predicted P(home win)')
ax.set_title('Strength gap vs predicted home-win probability'); ax.legend()
plt.tight_layout(); plt.show()

### Which features drive a home win?
Logistic regression coefficients for the **HW** class. Features are standardised inside the pipeline, so
the magnitudes are directly comparable. Positive = pushes toward a home win.

In [ ]:
clf = model.named_steps['clf']
hw_idx = list(clf.classes_).index('HW')
coefs = pd.Series(clf.coef_[hw_idx], index=X.columns).sort_values()
top = pd.concat([coefs.head(8), coefs.tail(8)])

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color=['#e76f51' if v < 0 else '#2a9d8f' for v in top.values])
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('coefficient (standardised)'); ax.set_title('Top drivers of a home win (LogReg, HW class)')
plt.tight_layout(); plt.show()

### Predicted home-win probability by actual outcome
A discrimination check: matches that really were home wins should have higher predicted `p_home`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for label in LABEL_ORDER:
    ax.hist(p_home[y_te.values == label], bins=20, alpha=0.6, label=label, color=colors[label])
ax.set_xlabel('predicted P(home win)'); ax.set_ylabel('matches')
ax.set_title('Predicted p_home split by actual outcome'); ax.legend()
plt.tight_layout(); plt.show()